# 🧪 W1-D3 概念实验：手写 Self-Attention 并"拷问"它

> 配套阅读：`第1周-Day3-代码实战与自注意力实现.md`（分步讲解在那边）
> 这个 notebook 把 Day1 的机制封装成函数，然后做 4 个"压力测试"：
> **置换等价性、数值稳定性、因果掩码、复杂度**——都是面试与实战的高频考点
>
> 实验环境：纯 numpy + matplotlib。

## 实验 1：实现可复用的 `self_attention()` 函数

把流程封装成函数（带可选 mask），并用两条不变式做单元测试：
① 输出 shape 与输入相同；② 权重每行和为 1。

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    """数值稳定的 softmax"""
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def self_attention(X, Wq, Wk, Wv, mask=None):
    """单头自注意力。X:(n,d) -> (n,d)。mask:(n,n)，1=可见，0=屏蔽"""
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask == 1, scores, -1e9)   # 屏蔽位给极小分数
    W = softmax(scores, axis=1)
    return W @ V, W

rng = np.random.default_rng(7)
X = rng.normal(size=(4, 8))              # 模拟 "我 爱 AI 学习"
Wq, Wk, Wv = (rng.normal(scale=0.5, size=(8, 8)) for _ in range(3))

out, W = self_attention(X, Wq, Wk, Wv)

assert out.shape == X.shape, "shape 不变"
assert np.allclose(W.sum(axis=1), 1.0), "每行是概率分布"
print("单元测试通过 ✓  out.shape =", out.shape, " 行和 =", W.sum(axis=1).round(6))

## 实验 2：置换等价性 —— 自注意力天生"看不见顺序"

把词序打乱再算，输出只是跟着换行，数值完全不变（逆置换还原后误差 ≈ 1e-16）。
**这不是 bug 是特性**：顺序信息必须由位置编码额外注入（Day2 的伏笔）。

In [ ]:
perm = rng.permutation(4)                    # 随机打乱词序
out_p, W_p = self_attention(X[perm], Wq, Wk, Wv)
inv = np.argsort(perm)                       # 逆置换，把行序还原

diff = np.abs(out_p[inv] - out).max()
print("打乱顺序:", perm, "（词序变成了 ['学习','AI','我','爱'] 之类）")
print(f"逆置换还原后与原输出的最大偏差: {diff:.2e}")
print("权重矩阵同样只是换行:", np.abs(W_p[inv] - W).max())
print("\n→ self_attention 本身是置换等变的：没有位置编码，'我打他' 和 '他打我' 无法区分")

## 实验 3：数值稳定性 —— 为什么 softmax 必须先减最大值？

大模型 logits 动辄上千。朴素 softmax 先算 exp(1000) 直接溢出为 inf，
得到 inf/inf = nan。减去每行最大值后，最大的指数项变成 exp(0)=1，永远安全。

In [ ]:
big = np.array([1000.0, 1001.0, 1002.0, 999.0])

with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(big)
    naive = naive / naive.sum()
print("朴素 softmax:", naive, " ← exp(1000) 溢出 → 全 nan")

stable = softmax(big)
print("稳定 softmax:", stable.round(6), " ← 最大分数减成 0 后再 exp")

# 再验证"平移不变性"：softmax(x + c) == softmax(x)，所以减最大值不改变结果
p1 = softmax(np.array([1.0, 2.0, 3.0]))
p2 = softmax(np.array([1.0, 2.0, 3.0]) + 1000.0)
print("\nsoftmax([1,2,3])          =", p1.round(6))
print("softmax([1,2,3] + 1000)   =", p2.round(6), " ← 平移不变，减 max 是免费的安全")

## 实验 4：因果掩码 —— 让每个词只能看左边

GPT 类生成模型的标配：下三角掩码。可视化掩码作用前后的权重矩阵，
右上三角（未来的词）权重被压成 0。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

n = 6
X6 = rng.normal(size=(n, 8))
causal = np.tril(np.ones((n, n), dtype=int))       # 下三角 = 只看左边(含自己)

_, W_free = self_attention(X6, Wq, Wk, Wv)               # 无掩码
_, W_causal = self_attention(X6, Wq, Wk, Wv, mask=causal)  # 因果掩码

print("因果掩码（1=可见）：\n", causal)
print("\n掩码后位置 4 的权重:", W_causal[4].round(4), " ← 位置 5 权重 =", round(W_causal[4, 5], 8))

toks = [f"词{i}" for i in range(n)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, mat, t in [(axes[0], W_free, "无掩码（BERT 式，双向全可见）"),
                   (axes[1], W_causal, "因果掩码（GPT 式，只看左边）")]:
    im = ax.imshow(mat, cmap="YlOrRd", vmin=0)
    ax.set_xticks(range(n), toks); ax.set_yticks(range(n), toks)
    ax.set_title(t); fig.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("因果掩码前后：右上三角（未来词）权重归零")
plt.tight_layout(); plt.show()

## 实验 5：复杂度 —— 为什么长上下文贵在注意力

自注意力的乘加量 ∝ L²·d（每对词都要打分），RNN/CNN ∝ L·d²。
画出来：序列长度超过 d（≈512）之后，注意力成为主导项，且随平方增长——
这正是后来 KV Cache / Flash Attention / 长文本优化要解决的问题（W2 伏笔）。

In [ ]:
Ls = np.array([256, 512, 1024, 2048, 4096, 8192, 16384, 32768])
d = 512
attn_flops = 2 * Ls**2 * d        # QK^T 与 WV 各一次：~2·L²·d
rnn_flops = 2 * Ls * d * d        # 逐步递归：~2·L·d²

cross = Ls[np.argmax(attn_flops > rnn_flops)]
print(f"注意力 FLOPs 在 L = {cross} 处超过 RNN（≈ d = {d}）")
print(f"L = 32768 时：注意力 {attn_flops[-1]/1e12:.1f} T-MACs，是 RNN 路线的 {attn_flops[-1]/rnn_flops[-1]:.0f} 倍")

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.loglog(Ls, attn_flops, "o-", label="自注意力  2·L²·d")
ax.loglog(Ls, rnn_flops, "s-", label="RNN/CNN  2·L·d²")
ax.axvline(cross, ls="--", color="gray", label=f"交叉点 L ≈ {cross}")
ax.set_xlabel("序列长度 L"); ax.set_ylabel("乘加次数 (MACs)")
ax.set_title("复杂度：注意力是 O(L²)，序列越长越是瓶颈")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 结论

| 考点 | 实验结论 |
|---|---|
| 函数封装 + 不变式 | shape 不变、行和=1，两条 assert 保驾（实验 1） |
| 置换等价性 | 自注意力无序 → 必须加位置编码（实验 2） |
| 数值稳定 | softmax 减 max 免费安全；朴素版 1000 就溢出（实验 3） |
| 因果掩码 | `-1e9` + softmax = 未来词权重归零（实验 4） |
| 复杂度 | O(L²·d)，L>d 后成为瓶颈 → 引出 W2 的推理优化（实验 5） |

→ 深入阅读：同目录 `.md` 版本第三节（逐步实现的分步讲解）